In [1]:
# IMPORT LIBRARIES
import os 
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
import pandas as pd 
from accelerate import Accelerator

current_dir  = os.path.abspath(os.getcwd())# Directorio actual
sys.path.append(os.path.dirname(current_dir))

from utils.data_load import WSI_Dataset, create_file_label_df, collate_fn
from utils.utils_train import run_epoch
from utils.heatmap_utils import build_heatmap, overlay_heatmap_on_image, draw_gt_patches

from models.keep_model import get_keep_ia3_model

# DEFINE PATHS 
ckpt_path = "/workspace/results/try_graph_0.0001_16_50_False/best_model"
result_path = "/workspace/data/patch_coord_2.xlsx"
folder_original_imgs = '/workspace/data/slides_filter'
num_img_show = 10

sicap_images_folder_path='/workspace/data/features_SICAP/patches'
sicap_labels_df_path= '/workspace/data/wsi_labels_changed_SICAP.xlsx'

# GENERAL CONFIGURATION 
print(" *Starting General Configuration ...")
CUDA_LAUNCH_BLOCKING=1
use_amp = torch.cuda.is_available()  # mixed precision if GPU available

accelerator = Accelerator()
device = accelerator.device
print(f" Usando dispositivo: {device}", flush = True)

# LOADING MODEL 
model = get_keep_ia3_model(clasification_model='graph', pretrain=False )
accelerator.load_state(ckpt_path)
model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)
print(" Pesos cargados correctamente")

loss = nn.BCELoss() 

# LOADING DATA
df_test = create_file_label_df(sicap_images_folder_path, sicap_labels_df_path, NAME = 'SICAP')

label_mapping = {0: 0, 1: 0, 2: 0, 3: 1, 4: 2, 5: 3}
df_test['gleason_primary_mapped'] = df_test['gleason_primary'].map(label_mapping)
df_test['gleason_secondary_mapped'] = df_test['gleason_secondary'].map(label_mapping)
df_test['gleason_score'] = df_test['gleason_score'].astype(int)

print("Loading embeddings for Inference ...")
inference_loader = DataLoader(WSI_Dataset(df_test, df_test.index, inference_heatmap=True), shuffle=False, batch_size=4, collate_fn=collate_fn)

# INFERENCE IN MODEL
model, inference_loader = accelerator.prepare(model, inference_loader)
inference_metrics, inference_loss, test_outputs= run_epoch(model, inference_loader, device, accelerator, loss, None, mode='test',inference_heatmap=True, y_prob_threshold=0.4, tumor_abs_threshold=0.1, alpha_pe=0, alpha_pc=0)   

# METRICS 
print(f"AUROC (Macro): {inference_metrics['multilabel']['auroc_macro']} \n")
print(f"AUROC (Weighted): {inference_metrics['multilabel']['auroc_weighted']} \n")
print(f"AUPRC (Macro): {inference_metrics['multilabel']['auprc_macro']} \n")
print(f"AUPRC (Weighted): {inference_metrics['multilabel']['auprc_weighted']} \n")

# COMAPRISON OF RESULTS
df_patch_class =  pd.read_excel(result_path)
test_outputs = test_outputs.head(num_img_show)

for index, row in test_outputs.iterrows():
    slide_id = row['slide_id']
    
    heatmap = build_heatmap(    row['coords'],
                                row['scores'], #row['weights']
                                patch_size=512,
                                scale_factor=16,
                                save_path=None)
    
    # Cargar la WSI original 
    slide_id = slide_id.replace('.h5','') + '.jpg'  # Ajustar extensión si es necesario
    file_name = slide_id.split('/')[-1]
    
    wsi_path = os.path.join(folder_original_imgs,file_name)
    print(f" Cargando WSI desde: {wsi_path} ")
    print(f" Scores de la slide {row['scores']} procesada.")
    orig = Image.open(wsi_path)
    w, h = orig.size
    new_size = (w // 16, h // 16)
    orig = orig.resize(new_size, Image.Resampling.BICUBIC)
    
    #Crear overlay heatmap sobre la imagen
    overlay = overlay_heatmap_on_image(orig, heatmap)

    # Dibujar parches GT en la imagen real (lado izquierdo)
    orig_with_gt = draw_gt_patches(orig.copy(), df_patch_class, file_name)

    # Crear imagen lado a lado
    w2, h2 = orig_with_gt.size
    canvas = Image.new("RGB", (w2*2, h2))
    canvas.paste(orig_with_gt, (0, 0))
    canvas.paste(overlay, (w2, 0))

    canvas.show()    

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True   n_gpus: 1
 *Starting General Configuration ...
 Usando dispositivo: cuda
trainable params: 0 || all params: 414,259,969 || trainable%: 0.0


ValueError: Tried to find /workspace/results/try_graph_0.0001_16_50_False/best_model but folder does not exist